# Chunks for TTS
One speaker per clip, 2-15 s, with text. Needs `04_diarization` for the same file.
Output: `outputs/chunks/<name>/wavs/` + `manifest.jsonl`.

In [23]:
import json
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np
import soundfile as sf
import nemo.collections.asr as nemo_asr

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
wavs = sorted((root / "data" / "raw_audios").glob("*.wav"))
for i, p in enumerate(wavs):
    print(i, p.name)

0 abuser.wav


In [24]:
audio_path = wavs[0]

MIN_SEC, TARGET_SEC, MAX_SEC = 2.0, 8.0, 15.0  # clip length
HARD_GAP_SEC = 1.0      # a pause this long always ends a clip
MIN_TURN_SEC = 0.2      # shorter diarization turns are noise
WHOLE_SENTENCES = True  # drop half sentences at the edges (if the text has punctuation)
MAX_CER = 0.35          # drop clips where the two transcripts disagree more than this

diar_dir = root / "outputs" / "diarization" / audio_path.stem / "nemotron"
rttm_path = diar_dir / f"{audio_path.stem}.rttm"
if not rttm_path.exists():
    raise FileNotFoundError(f"run 04_diarization on {audio_path.name} first")
out_dir = root / "outputs" / "chunks" / audio_path.stem
out_dir.mkdir(parents=True, exist_ok=True)

audio16, SR16 = sf.read(diar_dir / "audio_16k_mono.wav", dtype="float32")
total_sec = len(audio16) / SR16

# SPEAKER <file> 1 <start> <duration> <NA> <NA> <speaker> <NA> <NA>
turns = []
for line in rttm_path.read_text().splitlines():
    p = line.split()
    s, d = float(p[3]), float(p[4])
    if d >= MIN_TURN_SEC:
        turns.append((s, s + d, p[7]))
turns.sort()
print(audio_path.name, len(turns), "turns,", round(total_sec, 1), "s")

abuser.wav 19 turns, 69.2 s


In [25]:
# loudness every 10 ms, smoothed over 50 ms so short dips inside words don't count as pauses
hop = SR16 // 100
frames = audio16[: len(audio16) // hop * hop].reshape(-1, hop)
loud = np.convolve(np.sqrt((frames**2).mean(axis=1)), np.ones(5) / 5, mode="same")


def quietest(a, b):
    """Time (s) of the quietest moment between a and b."""
    i, j = max(int(a * 100), 0), min(int(b * 100), len(loud))
    if j - i < 2:
        return (a + b) / 2
    return (i + int(np.argmin(loud[i:j]))) / 100


WINDOW_SEC = 12  # the ASR was trained on clips up to 20 s; long pieces lose words and punctuation


def make_windows():
    """Pieces of at most WINDOW_SEC, each cut at the quietest moment of its last 4 s."""
    windows, start = [], 0.0
    while total_sec - start > WINDOW_SEC:
        cut = quietest(start + WINDOW_SEC - 4, start + WINDOW_SEC)
        windows.append((start, cut))
        start = cut
    windows.append((start, total_sec))
    return windows


windows = make_windows()
print(len(windows), "windows:", [(round(a, 1), round(b, 1)) for a, b in windows])

7 windows: [(0.0, 9.1), (9.1, 19.5), (19.5, 27.5), (27.5, 38.4), (38.4, 48.2), (48.2, 58.5), (58.5, 69.2)]


In [26]:
asr = nemo_asr.models.ASRModel.from_pretrained("nvidia/stt_uz_fastconformer_hybrid_large_pc")
asr.eval()


def transcribe(pieces, timestamps=False):
    """Run the ASR on (start, end) pieces of the 16 kHz audio."""
    with tempfile.TemporaryDirectory() as tmp:
        paths = []
        for k, (a, b) in enumerate(pieces):
            paths.append(f"{tmp}/{k:04d}.wav")
            sf.write(paths[-1], audio16[int(a * SR16):int(b * SR16)], SR16)
        hyps = asr.transcribe(paths, batch_size=4, timestamps=timestamps, verbose=False)
    return hyps[0] if isinstance(hyps, tuple) else hyps  # older NeMo returns (best, all)


words = []
for (a, _), h in zip(windows, transcribe(windows, timestamps=True)):
    ws = h.timestamp["word"]
    for k, w in enumerate(ws):
        text = w["word"]
        if k == len(ws) - 1:  # the model ends every piece with "." -> not a real sentence end
            text = text.rstrip(".?!…")
        if text:  # word times are counted from the window start -> add it
            words.append({"word": text, "start": round(a + w["start"], 3), "end": round(a + w["end"], 3)})
(out_dir / "words.json").write_text(json.dumps(words, ensure_ascii=False, indent=1), encoding="utf-8")
print(len(words), "words")
print(" ".join(w["word"] for w in words[:40]))

[NeMo I 2026-11-19 12:00:57 mixins:197] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-11-19 12:00:58 modelPT:192] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /workspace-mnt/final-train-manifests/MCV/train_manifest_final.json
    - /workspace-mnt/final-train-manifests/FLEURS/train_manifest_final.json
    - /workspace-mnt/final-train-manifests/UZBEKVOICE/train_manifest_final.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-11-19 12:00:58 modelPT:199] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the

[NeMo I 2026-11-19 12:00:58 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-11-19 12:00:58 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-19 12:00:58 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-11-19 12:00:58 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-19 12:00:58 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-11-19 12:00:58 save_restore_connector:309] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /Users/macbook/.cache/huggingface/hub/models--nvidia--stt_uz_fastconformer_hybrid_large_pc/snapshots/cd32f63d620a7b0cb461ed00ecfc45c852eb0bef/stt_uz_fastconformer_hybrid_large_pc.nemo.
[NeMo I 2026-11-19 12:00:58 hybrid_rnnt_ctc_models:148] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']
[NeMo I 2026-11-19 12:00:58 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-19 12:00:58 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available
[NeMo W 2026-11-19 12:00:58 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-19 12:00:58 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


71 words
Oʻzbekistonda navingni yetgan odam borligini bilarmidi hoziram keyingiski sekadigan odam bor Biroltichi boʻlgan professionallik bitta turnirda ismlari millatlarini Karachi ola Navini yutgan Navi bir olti paytida navin yutgan sekkan bitta xayla-etibor Nyuk koʻchada turadiday sayti boʻm boʻyotgani pastga oʻldirib qoʻyadi


In [27]:
def overlap(a1, b1, a2, b2):
    """Seconds shared by two time ranges."""
    return max(0.0, min(b1, b2) - max(a1, a2))


def word_speaker(w, near=0.3):
    """Speaker of a word, "OVERLAP" if two people talk during it, None if nobody is near."""
    talk = Counter()
    for s, e, spk in turns:
        talk[spk] += overlap(w["start"], w["end"], s, e)
    talk = +talk  # keep speakers with time > 0
    if len(talk) > 1 and sorted(talk.values())[-2] > 0.05:
        return "OVERLAP"
    if talk:
        return talk.most_common(1)[0][0]
    dist, spk = min(((max(s - w["end"], w["start"] - e), spk) for s, e, spk in turns), default=(near + 1, None))
    return spk if dist <= near else None


for i, w in enumerate(words):
    w["i"] = i
    w["speaker"] = word_speaker(w)
print(Counter(w["speaker"] for w in words))

Counter({'speaker_0': 70, 'OVERLAP': 1})


In [28]:
SENT_END = (".", "?", "!", "…")
CLAUSE_END = (",", ";", ":", "-", "—")


def split_runs(words):
    """Neighbouring words from ONE speaker, with no long pause or overlap inside."""
    runs, cur = [], []
    for w in words:
        bad = w["speaker"] in (None, "OVERLAP")
        if cur and (bad or w["speaker"] != cur[-1]["speaker"] or w["start"] - cur[-1]["end"] > HARD_GAP_SEC):
            runs.append(cur)
            cur = []
        if not bad:
            cur.append(w)
    if cur:
        runs.append(cur)
    return runs


def whole_sentences(run):
    """Start right after a sentence end, stop at a sentence end."""
    ends = [k for k, w in enumerate(run) if w["word"].endswith(SENT_END)]
    if not ends:
        return []
    i = run[0]["i"]
    clean_start = i == 0 or words[i - 1]["word"].endswith(SENT_END)
    return run[(0 if clean_start else ends[0] + 1):ends[-1] + 1]


runs = split_runs(words)
has_punct = sum(w["word"].endswith(SENT_END) for w in words) >= len(words) / 50
if WHOLE_SENTENCES and has_punct:
    runs = [r for r in map(whole_sentences, runs) if r]
elif WHOLE_SENTENCES:
    print("no sentence ends in the text -> cutting by pauses only")
print(len(runs), "runs")

no sentence ends in the text -> cutting by pauses only
8 runs


In [29]:
EDGE_SEC = 0.1  # rough silence at each clip edge, only used to estimate length


def cut_score(a, b):
    """How good a cut between word a and the next word b is. 7 at most."""
    pause = b["start"] - a["end"]
    score = 4 * min(pause, 1.0)
    if a["word"].endswith(SENT_END):
        score += 3
    elif a["word"].endswith(CLAUSE_END):
        score += 1
    if pause < 0.05:
        score -= 2
    return score


def best_cuts(run):
    """Dynamic programming: best[j] = lowest cost to split the first j words into clips."""
    n = len(run)
    best, prev = [0.0] + [float("inf")] * n, [0] * (n + 1)
    for j in range(1, n + 1):
        for i in range(j - 1, -1, -1):  # last clip = run[i:j]
            length = run[j - 1]["end"] - run[i]["start"] + 2 * EDGE_SEC
            if length > MAX_SEC and i < j - 1:
                break
            cost = best[i] + 4 * ((length - TARGET_SEC) / TARGET_SEC) ** 2
            if length < MIN_SEC:
                cost += 5
            if j < n:  # cutting after word j-1 (the end of the run is free)
                cost += 7 - cut_score(run[j - 1], run[j])
            if cost < best[j]:
                best[j], prev[j] = cost, i
    groups, j = [], n
    while j > 0:
        groups.append(run[prev[j]:j])
        j = prev[j]
    return groups[::-1]

In [30]:
QUIET = 0.3 * np.median(loud)  # quieter than this = nobody is speaking


def is_quiet(t):
    return loud[min(int(t * 100), len(loud) - 1)] <= QUIET


def make_clip(group):
    """Clip times for a group of words, or the reason it can't be used."""
    spk = group[0]["speaker"]
    while group:
        first, last = group[0], group[-1]
        prev_end = words[first["i"] - 1]["end"] if first["i"] > 0 else 0.0
        next_start = words[last["i"] + 1]["start"] if last["i"] + 1 < len(words) else total_sec
        # ASR word times are a bit off -> cut in the quietest moment next to the edge words
        start = quietest(max(prev_end, first["start"] - 0.4), first["start"])
        end = quietest(last["end"], min(next_start, last["end"] + 0.4))
        for s, e, other in turns:  # no other voice in the clip
            if other == spk or overlap(start, end, s, e) == 0:
                continue
            if e <= first["start"] + 0.05:
                start = max(start, e)
            elif s >= last["end"] - 0.05:
                end = min(end, s)
            else:
                return "other voice inside"
        # still loud at an edge = the ASR missed part of a word there -> drop that word, try again
        if not is_quiet(start):
            group = group[1:]
        elif not is_quiet(end):
            group = group[:-1]
        else:
            return {"start": round(start, 3), "end": round(end, 3), "speaker": spk,
                    "words": " ".join(w["word"] for w in group)}
    return "no quiet edge"


clips, dropped = [], Counter()
for run in runs:
    for group in best_cuts(run):
        c = make_clip(group)
        if isinstance(c, str):
            dropped[c] += 1
        elif c["end"] - c["start"] < MIN_SEC:
            dropped["too short"] += 1
        else:
            clips.append(c)
print(len(clips), "clips, dropped:", dict(dropped))

6 clips, dropped: {'too short': 2}


In [31]:
def clean_text(text):
    """One spelling for Uzbek apostrophes: oʻ gʻ with ʻ (U+02BB), tutuq belgisi with ʼ (U+02BC)."""
    out = []
    for k, ch in enumerate(text):
        if ch in "'‘’`ʼʻ":
            ch = "ʻ" if k > 0 and text[k - 1] in "oOgG" else "ʼ"
        out.append(ch)
    return " ".join("".join(out).split())


def cer(a, b):
    """Character error rate: edits needed to turn a into b, divided by the length of a."""
    a, b = (" ".join("".join(ch for ch in clean_text(t).lower() if ch.isalnum() or ch == " ").split()) for t in (a, b))
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1] / max(len(a), 1)


# second ASR pass on each clip alone: short pieces give the best text,
# and if it disagrees with the first pass, the clip is unreliable
for c, h in zip(clips, transcribe([(c["start"], c["end"]) for c in clips])):
    c["text"] = clean_text(h.text if hasattr(h, "text") else h)
    c["cer"] = round(cer(c["words"], c["text"]), 2)

for c in clips:
    keep = "keep" if c["cer"] <= MAX_CER else "DROP"
    print(f'{keep}  {c["start"]:6.2f}-{c["end"]:6.2f}  cer={c["cer"]:.2f}  {c["text"]}')
clips = [c for c in clips if c["cer"] <= MAX_CER]
print(len(clips), "clips,", round(sum(c["end"] - c["start"] for c in clips), 1), "s kept")

[NeMo I 2026-11-19 12:00:59 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-19 12:00:59 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo W 2026-11-19 12:00:59 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-19 12:00:59 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


keep    0.00-  2.82  cer=0.09  Oʻzbekistonda navin yutgan odami borligini bilarmidi?
keep    6.16- 10.73  cer=0.31  Hoziram keyskida oʻynaydi chekadigan odam borchi boʻlgan professionallik.
keep   15.17- 27.51  cer=0.16  Millatlarni Karachi ular navingni yutgan naviy bir olti paytida navin yutgan sekkan bitta xayr-eʼtibor Nyuk koʻchada torti boʻm oʻqiyotganini past oʻldirib qoʻyadi.
keep   38.37- 43.74  cer=0.09  U seki bilan gaplashganda aytganda Navi tepa qoʻl bermagan deganda yutqazib ketgandan keyin
keep   45.09- 48.51  cer=0.24  Hamma besh tashabbus ketadi-yu, yoqtirmayman.
keep   61.41- 69.22  cer=0.24  Zepeganlar bor ekan, kibersport tergi uchun oʻshalarni yutganch oʻzbek.
6 clips, 36.3 s kept


In [32]:
wav_dir = out_dir / "wavs"
wav_dir.mkdir(exist_ok=True)
for old in wav_dir.glob("*.wav"):
    old.unlink()

audio, sr = sf.read(audio_path, dtype="float32", always_2d=True)
audio = audio.mean(axis=1)  # cut from the original file, mono
fade = np.linspace(0, 1, int(0.01 * sr))  # 10 ms fade in/out, no clicks

with open(out_dir / "manifest.jsonl", "w", encoding="utf-8") as f:
    for n, c in enumerate(clips):
        x = audio[int(c["start"] * sr):int(c["end"] * sr)].copy()
        x[: len(fade)] *= fade
        x[-len(fade):] *= fade[::-1]
        name = f"{audio_path.stem}_{n:04d}_{c['speaker']}.wav"
        sf.write(wav_dir / name, x, sr)
        row = {"audio_filepath": f"wavs/{name}", "duration": round(len(x) / sr, 3), "text": c["text"],
               "speaker": f"{audio_path.stem}_{c['speaker']}"}  # speaker_0 is only unique inside one file
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("saved", len(clips), "clips to", wav_dir.relative_to(root))

saved 6 clips to outputs/chunks/abuser/wavs
